Databricks notebook source
MAGIC %md
MAGIC # Bronze data quality checks (alert only, never modifies data)

In [0]:
# COMMAND ----------
import uuid
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "clinical_dev")
dbutils.widgets.text("landing_path", "abfss://raw@landingstorageac.dfs.core.windows.net/clinical_trial/")
dbutils.widgets.text("max_staleness_hours", "36")

In [0]:
catalog = dbutils.widgets.get("catalog")
landing = dbutils.widgets.get("landing_path")
max_stale = int(dbutils.widgets.get("max_staleness_hours"))

In [0]:
table = f"{catalog}.bronze.clinical_trial_raw"
run_id = str(uuid.uuid4())
bronze = spark.read.table(table)
results = []

In [0]:
def record(name, severity, passed, value, threshold, details=""):
    results.append((run_id, "bronze", name, severity,
                    "PASS" if passed else "FAIL",
                    None if value is None else float(value), threshold, details))

In [0]:
# COMMAND ----------
# 1. Schema
EXPECTED = ["record_id","subject_id","trial_id","site_id","site_name","country","investigator_name",
            "age","gender","treatment_arm","visit_name","visit_date","weight_value","weight_unit",
            "height_cm","bmi","systolic_bp","diastolic_bp","heart_rate","hba1c_pct",
            "fasting_glucose_mg_dl","adherence_pct","ae_term","ae_severity","serious_ae_flag",
            "study_status","remarks","created_date","modified_date"]
AUDIT = {"_ingest_time", "_source_file", "_rescued_data"}

In [0]:
missing = [c for c in EXPECTED if c not in bronze.columns]
extra = [c for c in bronze.columns if c not in EXPECTED and c not in AUDIT]
record("expected_columns_present", "ERROR", not missing, len(missing), "0 missing", ",".join(missing))
record("no_unexpected_columns", "WARN", not extra, len(extra), "0 new columns", ",".join(extra))

In [0]:
types = dict(bronze.dtypes)
record("record_id_is_int", "ERROR", types.get("record_id") in ("int", "bigint"), None, "int", str(types.get("record_id")))
record("modified_date_is_timestamp", "ERROR", str(types.get("modified_date", "")).startswith("timestamp"), None, "timestamp", str(types.get("modified_date")))

In [0]:
if "_rescued_data" in bronze.columns:
    n = bronze.filter("_rescued_data IS NOT NULL").count()
    record("rescued_data_empty", "ERROR", n == 0, n, "0 rows")

In [0]:
# COMMAND ----------
# 2. Reconciliation: landing files vs Bronze rows, per file
landing_counts = (spark.read.option("recursiveFileLookup", "true").parquet(landing)
                  .select(F.col("_metadata.file_path").alias("_source_file"))
                  .groupBy("_source_file").count().withColumnRenamed("count", "landing_rows"))
bronze_counts = (bronze.groupBy("_source_file").count().withColumnRenamed("count", "bronze_rows"))
recon = (landing_counts.join(bronze_counts, "_source_file", "full_outer")
         .fillna(0, ["landing_rows", "bronze_rows"]))

In [0]:
not_loaded = recon.filter("landing_rows > 0 AND bronze_rows = 0").count()
mismatch   = recon.filter("landing_rows > 0 AND bronze_rows > 0 AND landing_rows <> bronze_rows").count()
orphans    = recon.filter("landing_rows = 0 AND bronze_rows > 0").count()
record("landing_files_loaded", "ERROR", not_loaded == 0, not_loaded, "0 files not loaded")
record("file_row_counts_match", "ERROR", mismatch == 0, mismatch, "0 mismatched files")
record("no_orphan_bronze_files", "WARN", orphans == 0, orphans, "0 files only in Bronze")

In [0]:
# COMMAND ----------
# 3. Keys and watermark column
total = bronze.count()
record("table_not_empty", "ERROR", total > 0, total, "> 0 rows")

In [0]:
null_ids = bronze.filter("record_id IS NULL").count()
dup_ids = bronze.groupBy("record_id").count().filter("count > 1").count()
record("record_id_not_null", "ERROR", null_ids == 0, null_ids, "0 nulls")
record("record_id_unique", "ERROR", dup_ids == 0, dup_ids, "0 duplicate ids", "duplicate record_id usually means a checkpoint reset or a double load")

In [0]:
null_mod = bronze.filter("modified_date IS NULL").count()
future_mod = bronze.filter("modified_date > current_timestamp() + INTERVAL 1 DAY").count()
record("modified_date_not_null", "ERROR", null_mod == 0, null_mod, "0 nulls")
record("modified_date_not_future", "WARN", future_mod == 0, future_mod, "0 rows")

In [0]:
# COMMAND ----------
# 4. Null rates on business keys
for col in ["subject_id", "visit_name", "visit_date"]:
    rate = bronze.filter(F.col(col).isNull()).count() / max(total, 1)
    record(f"{col}_null_rate", "WARN", rate <= 0.05, rate, "<= 5%")

In [0]:
# COMMAND ----------
# 5. Freshness
last_ingest = bronze.agg(F.max("_ingest_time")).first()[0]
age_h = spark.sql("SELECT 1").select(
    ((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp(F.lit(last_ingest))) / 3600).alias("h")
).first()["h"]
record("bronze_freshness_hours", "WARN", age_h <= max_stale, age_h, f"<= {max_stale} hours")

In [0]:
# COMMAND ----------
# Save the results, then fail the job if any ERROR check failed
schema = "run_id string, layer string, check_name string, severity string, status string, metric double, threshold string, details string"
res = spark.createDataFrame(results, schema).withColumn("checked_at", F.current_timestamp())
res.write.mode("append").saveAsTable(f"{catalog}.bronze.dq_results")
display(res.orderBy(F.col("status").desc(), "severity", "check_name"))

In [0]:
errors = res.filter("status = 'FAIL' AND severity = 'ERROR'").collect()
if errors:
    raise Exception("Bronze DQ failed: " + ", ".join(r.check_name for r in errors))